# 用 pymongo 操作 MongoDB

補充F 用 mongosh（要 `docker exec` 進容器）做的每一件事，這份 notebook 用 Python 做一遍。
概念完全相同：運算子字串一樣（`$gt`、`$group`、`$set`），差別只在語法外皮。

**前置**：
1. MongoDB 服務在跑：`docker compose -f docker-compose-local.yml up -d mongodb mongo-express`
2. 集合裡有資料：跑過補充F 的爬蟲（`uv run crawler/producer_crawler_finmind_mongo_single.py`）

對照說明與 mongo-express 介面版 → 補充F「同樣的操作」兩節

In [ ]:
# 連線：跟 crawler/tasks_crawler_finmind_mongo.py 用的是同一個套件、同一組參數
from pymongo import MongoClient

client = MongoClient(host="127.0.0.1", port=27017, username="root", password="1234")
col = client["mydb"]["TaiwanStockPrice"]   # 資料庫 mydb 裡的 TaiwanStockPrice 集合

col.count_documents({})   # 文件總數；{} 表示不設條件（SQL 的 SELECT COUNT(*)）

## 查詢

In [ ]:
# 條件 + 挑欄位 + 排序 + 限量（SQL：SELECT date, close ... WHERE ... ORDER BY date DESC LIMIT 3）
# 第一個參數是 filter（條件）、第二個是 projection（要哪些欄位；_id: 0 表示不要內建 id）
# .sort("date", -1)：-1 是由大到小（新→舊）、1 是由小到大
for d in col.find({"stock_id": "2330"}, {"_id": 0, "date": 1, "close": 1}).sort("date", -1).limit(3):
    print(d)

In [ ]:
# 集合裡有哪些股票（SQL 的 SELECT DISTINCT stock_id）
col.distinct("stock_id")

In [ ]:
# 範圍條件：收盤價 > 1000 的天數
# $gt = greater than；同一族還有 $gte（>=）、$lt（<）、$lte（<=）、$ne（!=）
col.count_documents({"stock_id": "2330", "close": {"$gt": 1000}})

In [ ]:
# 分組統計：每支股票的平均收盤價與筆數（SQL 的 GROUP BY + AVG + COUNT）
# aggregate 收一個「管線」：資料依序流過每個階段（$group 分組 → $sort 排序）
for d in col.aggregate([
    {"$group": {"_id": "$stock_id", "avg_close": {"$avg": "$close"}, "days": {"$sum": 1}}},
    {"$sort": {"_id": 1}},
]):
    print(d)

## 寫入與刪除

schema-free 的實際感受：insert 一份欄位完全不同的文件，MongoDB 照收（MySQL 會報 1054 拒絕）。

In [ ]:
# insert_one：寫入一份文件；欄位跟股價完全不同也不會被擋
r = col.insert_one({"stock_id": "TEST", "note": "欄位跟股價完全不同", "anything": [1, 2, 3]})
print("inserted_id:", r.inserted_id)   # 自動產生的 ObjectId——MongoDB 內建的代理鍵

col.find_one({"stock_id": "TEST"}, {"_id": 0})   # find_one：只取一份文件

In [ ]:
# delete_one：刪掉剛才的實驗文件（SQL 的 DELETE ... WHERE ... LIMIT 1）
col.delete_one({"stock_id": "TEST"}).deleted_count   # 回傳實際刪除的份數

## 防重複：unique index

MySQL 用主鍵擋重複；MongoDB 對應的機制是唯一索引。
我們的任務用 `update_one(upsert=True)` 本來就不會製造重複，unique index 是「保底」——
即使有程式用 insert 亂寫，資料庫層也擋得住。

In [ ]:
# 在 (stock_id, date) 上建唯一索引——對應 MySQL 的複合主鍵
# 1 表示升冪排序的索引方向；unique=True 是 mongo-express 介面做不到、要靠程式的部分
name = col.create_index([("stock_id", 1), ("date", 1)], unique=True)
print("index name:", name)

for i in col.list_indexes():   # 列出集合的所有索引
    print(dict(i))

In [ ]:
# 驗證：insert 同股同日的第二份文件會被唯一索引擋下
# DuplicateKeyError / E11000 就是 MongoDB 版的 MySQL 1062
from pymongo.errors import DuplicateKeyError

try:
    col.insert_one({"stock_id": "2330", "date": "2024-01-02", "close": 1})
except DuplicateKeyError as e:
    print("被擋下：", str(e)[:80])

In [ ]:
# 收尾：刪掉實驗索引、關閉連線
# （實務上這個索引可以留著當保底；教學收尾刪掉，讓環境回到原狀）
col.drop_index(name)
print("final count:", col.count_documents({}))   # 文件數跟一開始相同——所有實驗都清乾淨了
client.close()